# End to end — grounded work-order classification with Stirrup

Retrieve a real work order through MCP and classify its failure-code description without writing to
the database. The notebook discovers an existing work-order number first, so placeholders and missing
records cannot produce hallucinated classifications.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
TRACE_DIR = ARTIFACTS / "trajectories"
LOG_DIR = ARTIFACTS / "logs"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


In [ ]:
# Make the repository .env the source of truth for this tutorial run.
from dotenv import load_dotenv

ENV_FILE = REPO / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(
        f"Missing {ENV_FILE}. Copy .env.public to .env and configure a model provider."
    )
load_dotenv(ENV_FILE, override=True)
print("environment source:", ENV_FILE)
print("existing shell/kernel values overridden by .env: yes")

## 1. Create a read-only MCP client

`AOB_READONLY=1` removes work-order write tools from the server exposed to both this preflight and
Stirrup.


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_ENV = os.environ.copy()
MCP_ENV["AOB_READONLY"] = "1"

async def call_wo(tool_name, **arguments):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "wo-mcp-server"],
        cwd=str(REPO),
        env=MCP_ENV,
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Non-JSON response from {tool_name}: {text[:500]}") from exc
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(f"{tool_name} failed: {payload['error']}")
    return payload


## 2. Discover a real work order

The preferred TST benchmark record is used only if it exists. Otherwise, the notebook searches all
loaded sites and selects the first real record together with its actual site ID. This keeps the demo
grounded even when the TST scenario dataset has not been loaded.


In [ ]:
PREFERRED_SITE_ID = "TST"
PREFERRED_WONUM = "TST-WO00032"

listing = await call_wo(
    "list_workorders",
    page_size=0,
    page_num=1,
)
work_orders = [
    row for row in listing.get("work_orders", [])
    if row.get("wonum") and row.get("siteid")
]
if not work_orders:
    raise RuntimeError(
        "No work orders are loaded. Run: uv run python src/couchdb/init_data.py"
    )

preferred = next(
    (
        row for row in work_orders
        if row.get("siteid") == PREFERRED_SITE_ID
        and row.get("wonum") == PREFERRED_WONUM
    ),
    None,
)
selected = preferred or work_orders[0]
SITE_ID = str(selected["siteid"])
WORK_ORDER_NUMBER = str(selected["wonum"])

print("work orders found across all sites:", len(work_orders))
print("preferred TST work order present:", preferred is not None)
print("selected site:", SITE_ID)
print("selected work order:", WORK_ORDER_NUMBER)


In [ ]:
selected_record = await call_wo(
    "get_workorder",
    site_id=SITE_ID,
    wonum=WORK_ORDER_NUMBER,
)
selected_record


## 3. Configure Stirrup

Stirrup uses the model named by `KDD_MODEL_ID` in the repository `.env`. Supported routes are
`tokenrouter/...`, `litellm_proxy/...`, and `watsonx/...`. Choose a model known to execute native
structured tool calls. Credentials are checked without printing secret values.


In [ ]:
assert shutil.which("uv"), "Install uv first."
MODEL_ID = (os.getenv("KDD_MODEL_ID") or "").strip()
if not MODEL_ID:
    raise RuntimeError(
        f"Set KDD_MODEL_ID in {ENV_FILE}, then rerun the environment cell and this cell."
    )
if MODEL_ID.startswith("watsonx/"):
    required_credentials = ["WATSONX_APIKEY", "WATSONX_PROJECT_ID"]
elif MODEL_ID.startswith("tokenrouter/"):
    required_credentials = ["TOKENROUTER_API_KEY", "TOKENROUTER_BASE_URL"]
elif MODEL_ID.startswith("litellm_proxy/"):
    required_credentials = ["LITELLM_API_KEY", "LITELLM_BASE_URL"]
else:
    raise RuntimeError(
        "Unsupported KDD_MODEL_ID prefix. Use tokenrouter/, litellm_proxy/, or watsonx/. "
        f"Received: {MODEL_ID}"
    )
missing_credentials = [name for name in required_credentials if not os.getenv(name)]
print("agent framework: Stirrup")
print("model:", MODEL_ID)
print("credentials:", "ready" if not missing_credentials else "missing " + ", ".join(missing_credentials))
if MODEL_ID.startswith("watsonx/"):
    print("WARNING: the tested WatsonX model often serialized tool calls as text instead of executing them.")


## 4. Build the grounded question

The prompt contains a real number discovered above. It forbids simulation and all writes. If retrieval
fails, the required answer is `NOT FOUND`, preventing an unsupported classification.


In [ ]:
ALLOWED_DESCRIPTIONS = [
    "Breakdown",
    "Electrical",
    "Fail to function",
    "Leaking",
    "Low output",
    "Minor in-service problems",
    "Overheating",
    "Plugged / choked",
    "Structural deficiency",
    "Vibration",
]

QUESTION = f"""Call wo__get_workorder with exactly:
- site_id: "{SITE_ID}"
- wonum: "{WORK_ORDER_NUMBER}"

You must execute the tool. Never print, simulate, or assume a tool result.
Do not modify the site or work-order number.

If retrieval returns an error, output exactly:
NOT FOUND

Read the returned work-order description and failure-code field.

If an existing failure-code description is present, return it exactly as stored.
If the record contains a failure-code identifier instead of its description, use the
read-only wo__get_failure_codes tool to resolve its description.

If no failure code is recorded, select exactly one of these descriptions based only
on the retrieved work-order description:
{chr(10).join(ALLOWED_DESCRIPTIONS)}

Do not call generate_work_order, update_workorder, approve_workorder,
assign_technician, close_workorder, cancel_workorder, or any other write tool.

After retrieving the work order, your next response must contain exactly the selected
failure-code description, or NOT FOUND when retrieval failed.

Do not explain your selection.
Do not mention the work order or failure-code field.
Do not use quotes, Markdown, or additional text.
Call finish with reason set to exactly the same selected description.
"""

print(QUESTION)

## 5. Run Stirrup and persist the trajectory

Complete logs stay on disk. The notebook uses the persisted trajectory as the authoritative result,
so Rich CLI output cannot cause JSON or Jupyter IOPub failures.


In [ ]:
RUN_AGENT = True
AGENT_TIMEOUT_SECONDS = int(os.getenv("KDD_AGENT_TIMEOUT_SECONDS", "240"))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"kdd-stirrup-wo-{stamp}"
SCENARIO_ID = "kdd-workorder-failure-code-001"
trajectory_path = TRACE_DIR / f"{RUN_ID}.json"
stdout_path = LOG_DIR / f"{RUN_ID}.stdout.log"
stderr_path = LOG_DIR / f"{RUN_ID}.stderr.log"
RUN_COMPLETED = False

if missing_credentials:
    raise RuntimeError("Configure credentials and restart the kernel: " + ", ".join(missing_credentials))
if not RUN_AGENT:
    print("Agent skipped because RUN_AGENT=False.")
else:
    env = MCP_ENV.copy()
    env["AGENT_TRAJECTORY_DIR"] = str(TRACE_DIR)
    cmd = [
        "uv", "run", "--directory", str(REPO), "stirrup-agent",
        "--no-code", "--json", "--max-turns", "4",
        "--model-id", MODEL_ID,
        "--run-id", RUN_ID,
        "--scenario-id", SCENARIO_ID,
        QUESTION,
    ]
    try:
        completed = subprocess.run(
            cmd, env=env, text=True, capture_output=True,
            timeout=AGENT_TIMEOUT_SECONDS,
        )
        stdout_text = completed.stdout or ""
        stderr_text = completed.stderr or ""
        returncode = completed.returncode
    except subprocess.TimeoutExpired as exc:
        stdout_text = exc.stdout or ""
        stderr_text = exc.stderr or ""
        if isinstance(stdout_text, bytes):
            stdout_text = stdout_text.decode(errors="replace")
        if isinstance(stderr_text, bytes):
            stderr_text = stderr_text.decode(errors="replace")
        returncode = None
        print(f"Agent timed out after {AGENT_TIMEOUT_SECONDS}s; partial logs were saved.")

    stdout_path.write_text(stdout_text, encoding="utf-8")
    stderr_path.write_text(stderr_text, encoding="utf-8")

    if returncode not in (0, None):
        tail = "\n".join(stderr_text.splitlines()[-25:])
        print(f"Agent exited with {returncode}. Last stderr lines:\n{tail}")
    elif returncode == 0:
        RUN_COMPLETED = True

    if trajectory_path.exists():
        RUN_COMPLETED = True
        print("trajectory recovered:", trajectory_path)
    else:
        print("No trajectory was persisted. Try another configured tool-calling model.")
        print("stdout log:", stdout_path)
        print("stderr log:", stderr_path)


## 6. Load and inspect the trajectory

A **trajectory** is the structured execution record of one agent run. It captures the request, model, runner, ordered turns, tools that actually executed, their inputs and outputs, token usage, timing, and the final answer. This provides the provenance needed to debug, audit, replay, and evaluate the run instead of trusting only the final response.

| Field | What it contains |
| --- | --- |
| `run_id` | Unique execution identifier and persisted JSON filename. |
| `scenario_id` | Benchmark or tutorial scenario associated with the run. |
| `runner` and `model` | Agent implementation and model used. |
| `question` and `answer` | Complete prompt and final returned answer. |
| `trajectory.turns` | Ordered agent turns. |
| `tool_calls` | Tools actually executed—not calls merely written in model text. |
| `tool_calls[].input` and `output` | Exact tool arguments and returned evidence. |
| token and duration fields | Per-turn usage and elapsed time. |

The next cell loads the authoritative persisted JSON and prints it for inspection.


In [ ]:
if not trajectory_path.exists():
    raise FileNotFoundError(
        f"Trajectory not found: {trajectory_path}. Inspect {stdout_path} and {stderr_path}."
    )

trajectory = json.loads(trajectory_path.read_text(encoding="utf-8"))
print(json.dumps(trajectory, indent=2, ensure_ascii=False))


## 7. Audit grounding, safety, and output

The audit treats the persisted trajectory—not terminal display or model prose—as the source of truth. A run is accepted only when the exact work order was retrieved, its returned identifiers match, no write tool ran, retrieval preceded exactly one `finish` call, the answer is supported by stored or read-only-resolved data, and the final output is one line.

`NOT FOUND` is a valid agent failure response, but it is rejected for evaluation or leaderboard submission because no grounded classification was produced.


In [ ]:
def parse_tool_output(call):
    raw = call.get("output", {})
    if isinstance(raw, dict):
        return raw
    try:
        return json.loads(raw)
    except (TypeError, json.JSONDecodeError):
        return {"error": "non-JSON tool output"}

turns = trajectory.get("trajectory", {}).get("turns", [])
call_events = [
    {"position": (turn_position, call_position), "call": call}
    for turn_position, turn in enumerate(turns)
    for call_position, call in enumerate(turn.get("tool_calls", []))
]
call_names = [event["call"].get("name") for event in call_events]

retrieval_events = [
    event for event in call_events
    if event["call"].get("name") == "wo__get_workorder"
]
expected_input = {"site_id": SITE_ID, "wonum": WORK_ORDER_NUMBER}
retrieval_inputs_exact = bool(retrieval_events) and all(
    event["call"].get("input") == expected_input for event in retrieval_events
)
retrieval_output = parse_tool_output(retrieval_events[-1]["call"]) if retrieval_events else {}
retrieval_output_valid = (
    isinstance(retrieval_output, dict)
    and bool(retrieval_output)
    and not retrieval_output.get("error")
)
candidate_record = retrieval_output.get("work_order", {}) if retrieval_output_valid else {}
retrieved_record = candidate_record if isinstance(candidate_record, dict) else {}
retrieved_record_matches = bool(retrieved_record) and (
    str(retrieved_record.get("siteid")) == SITE_ID
    and str(retrieved_record.get("wonum")) == WORK_ORDER_NUMBER
)
retrieval_grounded = (
    retrieval_inputs_exact and retrieval_output_valid and retrieved_record_matches
)

write_tools = {
    "wo__generate_work_order", "wo__update_workorder", "wo__approve_workorder",
    "wo__assign_technician", "wo__close_workorder", "wo__cancel_workorder",
}
write_calls = [name for name in call_names if name in write_tools]

finish_events = [
    event for event in call_events if event["call"].get("name") == "finish"
]
one_finish_call = len(finish_events) == 1
retrieval_before_finish = bool(retrieval_events and finish_events) and (
    retrieval_events[-1]["position"] < finish_events[0]["position"]
)
answer = str(trajectory.get("answer", "")).strip()
finish_input = finish_events[0]["call"].get("input", {}) if one_finish_call else {}
finish_matches_answer = one_finish_call and (
    finish_input.get("answer") == answer and finish_input.get("reason") == answer
)

single_line_answer = bool(answer) and "\n" not in answer and "\r" not in answer
recorded_code = retrieved_record.get("failurecode") or retrieved_record.get("failure_code")
recorded_description = (
    retrieved_record.get("failure_code_description")
    or retrieved_record.get("failurecodedescription")
    or retrieved_record.get("failure_description")
)
resolution_descriptions = []
for event in call_events:
    if event["call"].get("name") != "wo__get_failure_codes":
        continue
    resolved = parse_tool_output(event["call"])
    if not isinstance(resolved, dict):
        continue
    for item in resolved.get("failure_codes", []):
        if str(item.get("code")) == str(recorded_code) and item.get("description"):
            resolution_descriptions.append(str(item["description"]).strip())

if recorded_description:
    answer_source_valid = answer == str(recorded_description).strip()
elif recorded_code:
    answer_source_valid = answer in resolution_descriptions
else:
    answer_source_valid = answer in ALLOWED_DESCRIPTIONS

failures = []
if not retrieval_events:
    failures.append("No executed wo__get_workorder call was recorded.")
elif not retrieval_inputs_exact:
    failures.append("A retrieval call used a different site or work-order number.")
elif not retrieval_output_valid:
    failures.append("The retrieval output was invalid JSON or reported an error.")
elif not retrieved_record_matches:
    failures.append("The retrieved record does not match the requested identifiers.")
if write_calls:
    failures.append(f"Forbidden write tools were executed: {write_calls}.")
if not one_finish_call:
    failures.append(f"Expected exactly one finish call; found {len(finish_events)}.")
elif not retrieval_before_finish:
    failures.append("The finish call did not occur after retrieval.")
elif not finish_matches_answer:
    failures.append("The finish answer/reason does not match the persisted answer.")
if not single_line_answer:
    failures.append("The final answer is empty or contains multiple lines.")
elif answer == "NOT FOUND":
    failures.append("The agent returned NOT FOUND, so the run is not grounded.")
elif not answer_source_valid:
    failures.append("The answer is not supported by retrieved or resolved failure-code data.")

audit = {
    "selected_work_order": f"{SITE_ID}/{WORK_ORDER_NUMBER}",
    "actual_tool_calls": call_names,
    "retrieval_call_count": len(retrieval_events),
    "retrieval_inputs_exact": retrieval_inputs_exact,
    "retrieved_record_matches": retrieved_record_matches,
    "retrieval_grounded": retrieval_grounded,
    "write_calls": write_calls,
    "retrieval_before_finish": retrieval_before_finish,
    "finish_call_count": len(finish_events),
    "finish_matches_answer": finish_matches_answer,
    "final_answer": answer,
    "single_line_answer": single_line_answer,
    "answer_source_valid": answer_source_valid,
    "failures": failures,
    "run_valid": not failures,
}
display(audit)


In [ ]:
if not audit["run_valid"]:
    print("RUN REJECTED: do not send this trajectory to evaluation or the leaderboard.")
    for failure in audit["failures"]:
        print("-", failure)
else:
    print("RUN ACCEPTED:", answer)
